# Sugidanon — Code-Switch Hiligaynon ASR Benchmark

Hiligaynon (Ilonggo): over 9 million speakers, nearly invisible to modern
speech technology. This notebook reproduces, on a fresh cloud machine in a
few minutes, the finding that off-the-shelf models hear the English and
Tagalog in Ilonggo speech but miss the Hiligaynon itself.

It downloads the open dataset, validates the benchmark, scores the bundled
baseline, and can rerun Whisper live to reproduce the switch penalty — word
error rate at the moment the language switches.

Dataset: https://huggingface.co/datasets/LauelKills/sugidanon-hil-codeswitch
Code: https://github.com/Jazztinn/tinig-sa-liwanag
Site: https://tinig-sa-liwanag.vercel.app

Runtime -> Run all. No setup. The bundled-baseline proof is quick; rerunning
Whisper takes a few minutes on the free CPU runtime.


## 1. Install


In [ ]:
!pip install -q openai-whisper huggingface_hub datasets
!apt-get -qq install -y ffmpeg >/dev/null


## 2. Get the code and dataset


In [ ]:
!git clone -q https://github.com/Jazztinn/tinig-sa-liwanag.git
%cd tinig-sa-liwanag
from huggingface_hub import snapshot_download
DS = snapshot_download('LauelKills/sugidanon-hil-codeswitch', repo_type='dataset')
print('dataset at:', DS)


## 3. Quick check: validate, score, and package the benchmark
This runs the documented release pipeline on the Hugging Face dataset using the repo's bundled baseline predictions, so it finishes fast. Section 4 then re-runs Whisper live so you can reproduce the same numbers from scratch — that live rerun is the real proof.


In [ ]:
!python scripts/build_release.py --annotations $DS/data/annotations --audio $DS/data/audio --predictions data/predictions --output /content/sugidanon_release --overwrite --skip-web


## 4. Optional live rerun: Whisper baseline
Whisper has no native Hiligaynon; Tagalog (tl) is the closest, which is the
point. Watch it transcribe the English and Tagalog words well and mangle the
Hiligaynon.


In [ ]:
!python scripts/run_whisper.py --audio-dir $DS/data/audio --out-dir /content/preds --model small --language tl


## 5. Score the live Whisper output
This should reproduce the same pattern as the bundled baseline: Hiligaynon
matrix words are harder than the borrowed Tagalog/English switch words.


In [ ]:
!python score.py --ref $DS/data/annotations --hyp /content/preds


## What you just saw

A negative switch penalty: words next to a language switch scored better than
the monolingual Hiligaynon. The switch regions carry the borrowed English and
Tagalog words the model already knows; the Hiligaynon matrix is what it cannot
handle. tl-en is near-solved (about 6 percent error); hil-en is the worst
(about 40 percent). The failure scales with how much Hiligaynon is in the
sentence — exactly what this dataset measures.

Swap `--model small` for `large-v3`, or run Meta MMS, to put another model on
the same benchmark. To compare your own ASR system, write one JSON prediction
per clip with `{ "clip_id": "hil_cs_001", "text": "..." }`, then run
`python score.py --ref $DS/data/annotations --hyp your_predictions_dir`.

**Limitations (read before citing numbers).** Preliminary: Whisper *small*, scripted elicited clips, one headline speaker, and a small benchmark size. Per-word language tags for the headline clips are speaker-reviewed. Treat this as a reproducible benchmark scaffold, not a final model ranking.
